# Install ComfyUI and dependencies

In [ ]:

# Install ComfyUI and dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q huggingface_hub
!mkdir -p /kaggle/tmp
!git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/tmp/ComfyUI
%cd /kaggle/tmp/ComfyUI
!pip install -q -r requirements.txt

# Install ComfyUI-Manager
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager

    
# Install ComfyUI-KJNodes (helper nodes used in the official LTX-2.3 GGUF workflow)
!git clone https://github.com/kijai/ComfyUI-KJNodes.git custom_nodes/ComfyUI-KJNodes
!pip install -q -r custom_nodes/ComfyUI-KJNodes/requirements.txt

# Install the missing GGUF loader custom node
!git clone https://github.com/city96/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
%cd /kaggle/tmp/ComfyUI/custom_nodes/ComfyUI-GGUF

# 1. Install required python libraries for GGUF parsing
!pip install -q -r requirements.txt
!pip install -q gguf sentencepiece

print("\nGGUF library dependencies successfully patched!")
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git custom_nodes/ComfyUI-VideoHelperSuite
# 1. Move into the video helper suite directory
%cd /kaggle/tmp/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite

# 2. Force install the required system and video manipulation libraries
!pip install -q -r requirements.txt
!pip install -q decord av simplejson imageio-ffmpeg

# Install cloudflared to expose the URL publicly
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


%cd /kaggle/tmp/ComfyUI

# Download models

In [ ]:
%cd /kaggle/tmp/ComfyUI

# 3. Create the required directory for clip_vision models
!mkdir -p models/clip_vision
# Create standard target directories
!mkdir -p models/unet
!mkdir -p models/text_encoders
!mkdir -p models/vae
!mkdir -p models/clip_vision


# Use huggingface_hub to fetch your chosen befox GGUF model paths
from huggingface_hub import hf_hub_download


print("Downloading clip_vision_h.safetensors from the correct repository...")
hf_hub_download(
    repo_id="Comfy-Org/Wan_2.1_ComfyUI_repackaged",  # Switched repo path
    filename="split_files/clip_vision/clip_vision_h.safetensors",
    local_dir="models/clip_vision",
    local_dir_use_symlinks=False
)

print("Downloading the correct Image-to-Video (I2V) GGUF file...")
hf_hub_download(
    repo_id="befox/WAN2.2-14B-Rapid-AllInOne-GGUF",
    filename="v10/wan2.2-i2v-rapid-aio-v10-nsfw-Q4_K.gguf", # Notice the i2v swap here
    local_dir="models/unet",
    local_dir_use_symlinks=False
)

print("Downloading Wan 2.1 VAE...")
hf_hub_download(
    repo_id="Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
    filename="split_files/vae/wan_2.1_vae.safetensors",
    local_dir="models/vae",
    local_dir_use_symlinks=False
)

print("Downloading Text Encoder...")
hf_hub_download(
    repo_id="Comfy-Org/Wan_2.1_ComfyUI_repackaged",
    filename="split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    local_dir="models/text_encoders",
    local_dir_use_symlinks=False
)

print("All set! Files successfully downloaded to /kaggle/tmp/ComfyUI/models/.")

# Install cloudflared for the tunnel public URL for the comfy UI

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/tmp/cloudflared
!chmod +x /kaggle/tmp/cloudflared
!ls -lh /kaggle/tmp/cloudflared

# Start Comfy UI

In [ ]:
import subprocess, time
    
# Added the --enable-manager flag and forced custom node checking
proc = subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8182", "--enable-manager"])
time.sleep(5)

# Create tunnel and get URL

In [ ]:
import subprocess

tunnel = subprocess.Popen(
    ["/kaggle/tmp/cloudflared", "tunnel", "--url", "http://localhost:8182"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Waiting for URL... (may take 15-20 seconds)")
for line in iter(tunnel.stdout.readline, b""):
    line = line.decode().strip()
    print(line)
    if "https://" in line and "trycloudflare.com" in line:
        for word in line.split():
            if word.startswith("https://") and "trycloudflare.com" in word:
                print("\n\u2705 Your ComfyUI URL:", word)
                break
        break